In [ ]:
from IPython.display import clear_output
clear_output(wait=True)
%pip install -q xgboost python-dotenv

Imports all required libraries and establishes a connection to the PostgreSQL database containing the liveability z-scores built in `Sydney_Liveability.ipynb`.

In [ ]:

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict, KFold, GroupKFold, cross_val_score

import xgboost as xgb

from dotenv import load_dotenv

load_dotenv("project.env")

db_url = os.getenv("DATABASE_URL")

engine = create_engine(db_url)


Sets up the SQLAlchemy database engine and defines the path to the local data directory containing the ABS Census files.

In [ ]:
data_dir = Path("data")

Loads the ABS 2021 Census G02 table (`2021Census_G02_NSW_SA2.csv`), which contains median weekly rent, income, mortgage repayments and household statistics for every SA2 region in NSW. This provides a clean, census-level target variable with complete coverage — eliminating the need for noisy property transaction data.

In [ ]:
df_census = pd.read_csv(data_dir / "2021Census_G02_NSW_SA2.csv" )

Retrieves the five liveability z-scores for all 360 Greater Sydney SA2 regions from the database, along with each SA2's parent SA4 region for geography-aware validation. The target variable is `Median_rent_weekly` — the median rent paid per week in each SA2 region according to the 2021 Census. SA2 regions with missing or zero rent values are excluded.

In [ ]:
query_ml_data = """
    SELECT 
        z.sa2_code,
        z.sa2_name,
        b.sa4_name,
        z.business_z, z.stop_z, z.school_z, z.poi_z, z.crime_z
    FROM z_scores z
    JOIN sa2_boundaries b ON z.sa2_code = b.sa2_code;
"""
df_ml = pd.read_sql(query_ml_data, engine)
df_ml['sa2_code'] = df_ml['sa2_code'].astype('int64')
df_ml = df_ml.merge(df_census, left_on='sa2_code', right_on='SA2_CODE_2021', how='inner')
df_ml = df_ml.dropna(subset=['Median_rent_weekly'])
df_ml = df_ml[df_ml['Median_rent_weekly'] > 0]
print(f"Ready for ML with {len(df_ml)} SA2s.")
print(df_ml['Median_rent_weekly'].describe())

Trains two regression models — **Random Forest** and **XGBoost** — using the five liveability z-scores as features and median weekly rent as the target. XGBoost uses `max_depth=2` to prevent overfitting on this small 360-row dataset. Results shown here come from a single 80/20 train-test split (`random_state=42`) and are illustrative only; the validation cells that follow provide the headline estimates.

In [ ]:
features = ['business_z', 'stop_z', 'school_z', 'poi_z', 'crime_z']
X = df_ml[features]
y = df_ml['Median_rent_weekly']
groups = df_ml['sa4_name']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

xgb_model = xgb.XGBRegressor(
    n_estimators=190, 
    max_depth=2, 
    learning_rate=0.05,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

models = {"Random Forest": rf_pred, "XGBoost": xgb_pred}
for name, preds in models.items():
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"{name:<15} | R²: {r2:.3f} | RMSE : {rmse:.3f}")

Evaluates both models with two complementary validation schemes. Shuffled 5-fold cross-validation estimates how well the models interpolate across the existing Sydney SA2 mix, while GroupKFold by SA4 is a stricter geography-aware holdout that tests whether relationships learned in one part of Sydney transfer to unseen subregions. The first metric is useful for within-city interpolation; the second is the better proxy for out-of-region generalisation.

In [ ]:

shuffled_kf = KFold(n_splits=5, shuffle=True, random_state=42)
grouped_kf = GroupKFold(n_splits=5)

print("      SHUFFLED 5-FOLD CROSS-VALIDATION    ")

rf_cv_r2 = cross_val_score(rf_model, X, y, cv=shuffled_kf, scoring='r2')
xgb_cv_r2 = cross_val_score(xgb_model, X, y, cv=shuffled_kf, scoring='r2')

rf_cv_rmse = np.sqrt(-cross_val_score(rf_model, X, y, cv=shuffled_kf, scoring='neg_mean_squared_error'))
xgb_cv_rmse = np.sqrt(-cross_val_score(xgb_model, X, y, cv=shuffled_kf, scoring='neg_mean_squared_error'))

print(f"{'Model':<15} | {'Average R²':<15} | {'Average RMSE'}")
print("-" * 50)
print(f"Random Forest   | {rf_cv_r2.mean():.3f} (±{rf_cv_r2.std():.3f})  | ${rf_cv_rmse.mean():.2f}")
print(f"XGBoost         | {xgb_cv_r2.mean():.3f} (±{xgb_cv_r2.std():.3f})  | ${xgb_cv_rmse.mean():.2f}")

print()
print("      GEOGRAPHY-AWARE GROUPKFOLD BY SA4   ")

rf_group_cv_r2 = cross_val_score(rf_model, X, y, cv=grouped_kf, groups=groups, scoring='r2')
xgb_group_cv_r2 = cross_val_score(xgb_model, X, y, cv=grouped_kf, groups=groups, scoring='r2')

rf_group_cv_rmse = np.sqrt(-cross_val_score(rf_model, X, y, cv=grouped_kf, groups=groups, scoring='neg_mean_squared_error'))
xgb_group_cv_rmse = np.sqrt(-cross_val_score(xgb_model, X, y, cv=grouped_kf, groups=groups, scoring='neg_mean_squared_error'))

print(f"{'Model':<15} | {'Average R²':<15} | {'Average RMSE'}")
print("-" * 50)
print(f"Random Forest   | {rf_group_cv_r2.mean():.3f} (±{rf_group_cv_r2.std():.3f})  | ${rf_group_cv_rmse.mean():.2f}")
print(f"XGBoost         | {xgb_group_cv_r2.mean():.3f} (±{xgb_group_cv_r2.std():.3f})  | ${xgb_group_cv_rmse.mean():.2f}")

Plots Random Forest feature importance (mean decrease in impurity) to show which liveability factors most influence predicted rent. Random Forest is used here rather than XGBoost because its importance scores spread more evenly across features, giving a cleaner picture of each factor's individual contribution.

In [ ]:
importances = rf_model.feature_importances_
feature_names = ['Business Density', 'Transit Access', 'School Capacity', 'Points of Interest', 'Crime Rate']

df_importance = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
df_importance = df_importance.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=df_importance, palette='viridis', hue='Feature', legend=False)
plt.title('What Drives Sydney Rents? (Random Forest Feature Importance, Liveability Only)', fontsize=14, pad=15)
plt.xlabel('Relative Importance', fontsize=12)
plt.ylabel('')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print(df_importance.to_string(index=False))

Plots predicted versus actual median weekly rent for both models on the held-out test set (72 SA2 regions). Each point represents one SA2 region — points lying on the diagonal dashed line indicate a perfect prediction. 

Scatter above the line means the model under-predicted rent; scatter below means it over-predicted. The tighter the cloud around the diagonal, the stronger the model's predictive signal from the five liveability z-scores.

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
models = [('Random Forest', rf_pred), ('XGBoost', xgb_pred)]

for ax, (name, preds) in zip(axes, models):
    sns.scatterplot(x=y_test, y=preds, alpha=0.5, edgecolor='k', ax=ax)
    
    ax.axline((0, 0), slope=1, color='r', linestyle='--', linewidth=2, label='Perfect Prediction')
    
    ax.set(title=f'{name} (R² = {r2_score(y_test, preds):.3f})', 
           xlabel='Actual Median Weekly Rent ($)', 
           ylabel='Predicted Median Weekly Rent ($)')
    ax.legend()
    ax.set_xlim(left=300)
    ax.set_ylim(bottom=300)

plt.suptitle('Predicted vs Actual Rent: Single Split', fontsize=15, y=1.02)
plt.tight_layout()

plt.show()


Positive arbitrage (predicted > actual) identifies suburbs where the model's within-Sydney liveability signal sits above the observed rent, while negative arbitrage identifies suburbs where rent is higher than the liveability-only model would expect. Because geography-aware validation remains much weaker than shuffled validation, these values should be treated as exploratory within-city signals rather than deployable forecasts of future rental growth.

In [ ]:

oof_predictions = cross_val_predict(xgb_model, X, y, cv=shuffled_kf)

df_ml['predicted_rent'] = oof_predictions
df_ml['actual_rent'] = df_ml['Median_rent_weekly']
df_ml['arbitrage'] = df_ml['predicted_rent'] - df_ml['actual_rent']

results_df = df_ml[['sa2_name', 'actual_rent', 'predicted_rent', 'arbitrage']].copy()
results_df.to_csv('data/arbitrage_results.csv', index=False)